# 02 - Preprocessing: Stationarity, Feature Matrix, Train/Val/Test Split

Task 2.4-2.5 and 3.1: ADF/KPSS/ARCH-LM diagnostics, the leakage-safe feature matrix (Listing 2.3), and the strict chronological 70/15/15 split (Listing 3.1).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config

In [ ]:
from src import data, diagnostics, features

df = data.load_or_download(synthetic=False)

## Stationarity: level vs returns (sec. 2.5, Listing 2.4)

Expect log-price to fail to reject the ADF null (unit root) and reject KPSS stationarity; returns should do the opposite.

In [ ]:
for name, s in [("log_close", df["log_close"]), ("ret", df["ret"])]:
    print(diagnostics.stationarity_tests(s, name))

In [ ]:
arch_lm = diagnostics.arch_lm_test(df["ret"])
print("ARCH-LM p-value (expect ~0):", arch_lm["lm_p"])

## Feature matrix (Listing 2.3)

Every column is shifted so only information available strictly before the target date is used - see `src/features.py` for the exact contract, including the deliberate extra shift on the calendar indicators.

In [ ]:
feat = features.build_feature_matrix(df)
print(feat.shape)
feat.head()

## Chronological 70/15/15 split (Listing 3.1)

In [ ]:
train, val, test = features.chronological_split(feat)
for name, block in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} n={len(block):5d}  {block.index.min().date()} -> {block.index.max().date()}")

In [ ]:
# Persist for the other notebooks / scripts to reuse.
df.to_parquet(config.PROCESSED_DATA_DIR / "prices_clean.parquet")
feat.to_parquet(config.PROCESSED_DATA_DIR / "features_full.parquet")
train.to_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val.to_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test.to_parquet(config.PROCESSED_DATA_DIR / "test.parquet")